In [4]:
!apt-get install -y ffmpeg
!pip install ultralytics yt-dlp filterpy opencv-python huggingface_hub roboflow -q
import cv2
import os
import numpy as np
from ultralytics import YOLO
from filterpy.kalman import KalmanFilter
from huggingface_hub import hf_hub_download
from google.colab.patches import cv2_imshow

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [5]:
from roboflow import Roboflow
import os
rf = Roboflow(api_key="xxxxxxxxxxxxxxx")

try:
    project = rf.workspace("yolov8-drone-detection").project("yolov8_detfly-02")
    dataset = project.version(1).download("yolov8")
    DATASET_YAML = os.path.join(dataset.location, "data.yaml")
    print(f"Success: {dataset.location}")
except Exception as e:
    print(f"Error: {e}")

loading Roboflow workspace...
loading Roboflow project...
✅ Success! Dataset downloaded to: /content/YOLOv8_DetFly(02)-1


In [3]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=DATASET_YAML,
    epochs=25,
    imgsz=640,
    batch=16,
    name='drone_detector_v1',
    device=0
)

TRAINED_WEIGHTS = "/content/runs/detect/drone_detector_v1/weights/best.pt"
print(f"Weights saved at: {TRAINED_WEIGHTS}")

Ultralytics 8.4.24 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLOv8_DetFly(02)-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=drone_detector_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

In [11]:
os.makedirs('detections', exist_ok=True)
MODEL_PATH = '/content/runs/detect/drone_detector_v1/weights/best.pt'
model = YOLO(MODEL_PATH)
base_dir = "/content/drive/MyDrive/CS-UY6613 Assignment 3"
import glob

In [9]:
#for processing all videos
def process_all_videos(directory):
    video_files = glob.glob(os.path.join(directory, "*.mp4"))

    if not video_files:
        print("No .mp4 files found")
        video_files = glob.glob(os.path.join(directory, "*/*.mp4"))

    for video_path in video_files:
        video_name = os.path.basename(video_path).split('.')[0]
        print(f"Processing Video: {video_name}")

        cap = cv2.VideoCapture(video_path)
        frame_count = 0
        saved_count = 0

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frame_count += 1
            results = model.predict(frame, conf=0.4, verbose=False)
            if len(results[0].boxes) > 0:
                save_path = f"detections/{video_name}_frame_{frame_count:04d}.jpg"
                cv2.imwrite(save_path, frame)
                saved_count += 1
        cap.release()
        print(f"Finished {video_name}: Scanned {frame_count} frames, saved {saved_count} detections.")
process_all_videos(base_dir)

NameError: name 'glob' is not defined

In [12]:
#for already extracted
frame_folders = [
    "/content/drive/MyDrive/CS-UY6613 Assignment 3/frames/video_1",
    "/content/drive/MyDrive/CS-UY6613 Assignment 3/frames/video_2"
]

for folder in frame_folders:
    folder_name = os.path.basename(folder)
    print(f"Processing frame folder: {folder_name}")
    images = glob.glob(os.path.join(folder, "*.jpg"))

    for img_path in images:
        img = cv2.imread(img_path)
        res = model.predict(img, conf=0.4, verbose=False)

        if len(res[0].boxes) > 0:
            fname = os.path.basename(img_path)
            cv2.imwrite(f"detections/{folder_name}_{fname}", img)

Processing frame folder: video_1
Processing frame folder: video_2


In [14]:
from filterpy.kalman import KalmanFilter
import numpy as np

def init_kalman(initial_x, initial_y):
    kf = KalmanFilter(dim_x=4, dim_z=2)
    kf.x = np.array([initial_x, initial_y, 0, 0])
    kf.F = np.array([[1, 0, 1, 0],
                     [0, 1, 0, 1],
                     [0, 0, 1, 0],
                     [0, 0, 0, 1]])
    kf.H = np.array([[1, 0, 0, 0],
                     [0, 1, 0, 0]])

    kf.P *= 1000.
    kf.R = np.eye(2) * 5
    kf.Q = np.eye(4) * 0.1
    return kf

In [28]:
import cv2
import os
from ultralytics import YOLO

model = YOLO('/content/runs/detect/drone_detector_v1/weights/best.pt')
video_inputs = ["/content/drive/MyDrive/CS-UY6613 Assignment 3/drone_video_1.mp4", "/content/drive/MyDrive/CS-UY6613 Assignment 3/drone_video_2.mp4"]

def track_drone_in_video(video_path, output_name):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: {video_path}")
        return

    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    out = cv2.VideoWriter(output_name, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

    kf = None
    trajectory = []

    print(f"Starting: {video_path}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        if kf is not None:
            kf.predict()

        results = model(frame, conf=0.5, verbose=False)
        boxes = results[0].boxes.xyxy.cpu().numpy()

        if len(boxes) > 0:
            box = boxes[0]
            cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2

            if kf is None:
                kf = init_kalman(cx, cy)
            else:
                kf.update([cx, cy])

            cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), (0, 255, 0), 2)

        if kf is not None:
            est_x, est_y = int(kf.x[0]), int(kf.x[1])
            trajectory.append((est_x, est_y))

            if len(trajectory) > 1:
                for i in range(1, len(trajectory)):
                    cv2.line(frame, trajectory[i-1], trajectory[i], (0, 0, 255), 2)

            out.write(frame)

    cap.release()
    out.release()
    print(f"Video saved: {output_name}")

# --- UPDATED NAMING LOGIC ---
for vid in video_inputs:
    # os.path.basename extracts just 'drone_video_1.mp4' from the long path
    clean_name = os.path.basename(vid)
    output_filename = f"/content/tracked_{clean_name}"
    track_drone_in_video(vid, output_filename)

Starting: /content/drive/MyDrive/CS-UY6613 Assignment 3/drone_video_1.mp4
Video saved: /content/tracked_drone_video_1.mp4
Starting: /content/drive/MyDrive/CS-UY6613 Assignment 3/drone_video_2.mp4
Video saved: /content/tracked_drone_video_2.mp4


In [29]:
import subprocess

raw_outputs = [f"/content/tracked_{os.path.basename(vid)}" for vid in video_inputs]

for raw_v in raw_outputs:
    clean_name = raw_v.replace(".mp4", "_final.mp4")

    print(f"Finishing {os.path.basename(clean_name)} w/ FFmpeg")

    subprocess.run([
        'ffmpeg', '-y', '-i', raw_v,
        '-c:v', 'libx264', '-crf', '23',
        '-preset', 'fast', clean_name
    ])

    print(f"Final ready: {clean_name}")

from google.colab import files
for vid in video_inputs:
    final_file = f"/content/tracked_{os.path.basename(vid)}".replace(".mp4", "_final.mp4")
    if os.path.exists(final_file):
        files.download(final_file)
        files.download(final_file)

🎬 Finishing tracked_drone_video_1_final.mp4 w/ FFmpeg
✅ Final deliverable ready: /content/tracked_drone_video_1_final.mp4
🎬 Finishing tracked_drone_video_2_final.mp4 w/ FFmpeg
✅ Final deliverable ready: /content/tracked_drone_video_2_final.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import pandas as pd
from PIL import Image
import io

def create_hf_dataset(img_folder, output_parquet):
    data = []
    valid_extensions = ('.jpg', '.jpeg', '.png')
    print(f"Processing images{img_folder}")
    for filename in os.listdir(img_folder):
        if filename.lower().endswith(valid_extensions):
            img_path = os.path.join(img_folder, filename)
            with open(img_path, "rb") as f:
                img_bytes = f.read()
            data.append({
                "image": {"path": filename, "bytes": img_bytes},
                "file_name": filename
            })
    df = pd.DataFrame(data)
    df.to_parquet(output_parquet)
    print(f"Dataset saved to {output_parquet}")
create_hf_dataset('detections', 'drone_detections.parquet')

Processing imagesdetections
Dataset saved to drone_detections.parquet
